In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
class Info:
    """ Clase para almacenar información acerca del registro de datos"""

    def __init__(self, experimenter:str, subject_info:dict, ch_names:list | str, ch_types:list | str, bads:list | str, description:str, fm:float = 512):
        """
            Genera un objeto Info()
            Args:
                experimenter : Nombre del experimentador.
                subject_info : Información adicional del sujeto.
                ch_names : Lista con los nombres de los canales.
                ch_types : Tipo de cada canal ('emg', 'eeg', 'ecg') o un único tipo para todos.
                bads : Lista de canales marcados como "malos ".
                fm : Frecuencia de muestreo en Hz (por defecto 512).
                description : Descripción del registro de datos."""
        
        if len(ch_names) != len(ch_types):
            raise ValueError ("La cantidad de canales y los tipos de canales deben tener la misma longuitud")
        
        self.data = {"Experimentador": experimenter,
                     "Sujeto":subject_info,
                     "Nombre canales": ch_names,
                     "Tipo canales": ch_types,
                     "Canales malos": bads,
                     "Descripción": description,
                     "Frecuencia muestreo": fm}        

    def __contains__(self, clave):
        """ Permite verificar si una clave esta presente en el objeto
            Args:
                clave : Nombre de la clave que se quiere verificar
            Returns:
                True: La clave se encuentra en el objeto
                False: La clave no se encuentra en el objeto"""
        
        if clave in self.data:
            return True
        else:
            return False
        
    def __getitem__(self, clave):
        """ Permite acceder a elementos como un diccionario
            Args
                clave: Nombre de la clave a la que se quiere acceder
            Returns
                Devuelve el valor asociado a la clave ingrsada, o
                False: Cuando la clave ingresada no existe """
        
        if self.__contains__(clave) == True:
            return self.data[clave]
        else:
            return False
        
    def __len__(self) -> int:
        """ Devuelve la cantidad de elementos almacenados"""
        return len(self.data)
    
    def keys(self):
        """ Devuelve las claves del objeto"""
        return [clave for clave in self.data]
    
    def get(self, clave):
        """ Obtiene solo el valor de una clave específica
            Args:
                clave : Clave de la cual se quiere conocer su valor
            Returns:
                Devuleve el valor de la clave si la misma existe, o false en caso de que no exista"""
        
        if self.__contains__(clave) == True:
            return self.data[clave]
        else:
            return False

    def item(self, elemento):
        """ Devuelve los elementos como pares clave-valor de una clave en específica
            Args
                elemento : Elemento del que se quiere obtener la clave y su valor
            Returns:
                tuple : Tupla que contiene la clave y el valor del elemento
                False : Si no exite la clave en el diccionario"""
        
        valor = self.__getitem__(elemento)
        if valor == False:
            return False
        else:
            return (elemento, valor)
        
    def _check_channel(self, channel_name):
        """ Verifica si un canal se encuentra entre los nombres de los canales
            Args:
                channel_name : Canal a verificar si se encuentra en la lista de canales"""
        
        if channel_name in self.data["Nombre canales"]:
            return True
        else:
            return False
        
    def rename_channels(self, nombre_canal, nuevo_nombre):
        """Permite renombrar canales de forma segura
            Args:
                nombre_canal : Nombre del canal que se quiere cambiar
                nuevo_nombre : Nuevo nombre que va a tener el canal
            Returs:
                True : Si se cambio el nombre del canal correctamente
                False: Si no se pudo modificar el nombre del canal"""
        
        if self._check_channel(nombre_canal) == True:
            indice = self.data["Nombre canales"].index(nombre_canal)
            self.data["Nombre canales"][indice] = nuevo_nombre
            return True
        else:
            return False
    
    def eliminar_elementos(self, key:str, elementos:str|list):
        """
        Elimina uno o varios elementos de la lista asociada a una clave específica.

        Args:
            clave : Clave del diccionario donde se eliminarán los elementos.
            elementos : Elemento o lista de elementos a eliminar.

        Raises:
            KeyError: Si la clave no existe en el diccionario.
            ValueError: Si uno o más elementos a eliminar no están presentes en la lista."""
        
        if self.__contains__(key) == False:
            raise KeyError(f"La clave '{key}' no existe en Info.")
        
        
        for elemento in elementos:
            if elemento not in self.data[key]:
                raise ValueError(f"El elemento '{elemento}' no se encuentra en la lista asociada a '{key}'.")
            self.data[key].remove(elemento)

In [4]:
canales = ["F2", "F3"]
tipos_canales = ["ecg"] * len(canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

In [5]:
info = Info("Juan Acosta", sujeto, canales, tipos_canales, None, "Pruebita", 512)
# print(info.__contains__("Experimentador"))

print("Nombre del experimentador:",info.__getitem__("Experimentador"))

print(info.__len__())  # Cantidad de elementos almacenados

print(info.keys()) 

print(info.item("Experimentador"))
print(info.item("Nombre canales"))

Nombre del experimentador: Juan Acosta
7
['Experimentador', 'Sujeto', 'Nombre canales', 'Tipo canales', 'Canales malos', 'Descripción', 'Frecuencia muestreo']
('Experimentador', 'Juan Acosta')
('Nombre canales', ['F2', 'F3'])


In [6]:
# Cambio el nombre de un canal
info.rename_channels(2, 5)
print(info.item("Nombre canales"))

('Nombre canales', ['F2', 'F3'])


In [7]:
# Eliminar elementos de una clave (en este caso de los canales)
info.eliminar_elementos(key="Nombre canales", elementos=["F3"])

In [8]:
print(info.item("Nombre canales"))

('Nombre canales', ['F2'])


In [9]:
class Anotaciones:
    """ Almacena y gestiona información relacionada con eventos en registros fisiológicos. 
        Permite la adición, eliminación y modificación de eventos."""
    
    def __init__(self, onset:np.ndarray, duration:np.ndarray, description):
        """Inicializa la clase con los datos de las anotaciones"""
        self.onset = onset
        self.duration = duration
        self.description = description

        if len(onset) != len(duration):
            raise ValueError ("Onset y duration deben tener la misma cantidad de elementos")
        
        self.anotaciones = pd.DataFrame({"Inicio (s)": self.onset, "Duración (s)": self.duration, "Descripcion": self.description})

    def get_annotations(self):
        """Devuelve una DataFrame con todas las anotaciones que recibe el constructor"""
        return self.anotaciones
    
    def add(self, anotacion:list|tuple):
        """Agrega una nueva anotación
            Args:
                anotacion : Nueva anotación a agregar (se espera la forma [inicio, duración, descripción])"""
        if len(anotacion) != 3:
            return False 
        else:
            self.anotaciones.loc[len(self.anotaciones)] = anotacion  # Agrega un fila
            return True 
    
    def remove(self, anotacion_eliminar):
        """Elimina una anotación específica
            Args:
                anotacion_eliminar : Anotación que se quiere eliminar
            Returns:
                True : Selimino correctamente la anotación
                False : No se pudo eliminar la anotación """
        
        if len(anotacion_eliminar) != 3:
            return False
        else:
            eliminar = (self.anotaciones["Inicio (s)"] == anotacion_eliminar[0]) & (self.anotaciones["Duración (s)"] == anotacion_eliminar[1]) & (self.anotaciones["Descripcion"] == anotacion_eliminar[2])
            indice = self.anotaciones[eliminar].index
            self.anotaciones = self.anotaciones.drop(indice)
            return True

    def find(self, buscar_anotacion):
        """Busca y devuelve una anotación específica
            Args:
                buscar_anotacion : Anotación que se quiere buscar entre los datos 
            Returs:
                Devuelve la anotación o False si la longuitud de la anotación no coincide con la estructura de los datos"""
        
        if len(buscar_anotacion) != 3:
            return False
        else:
            buscar = (self.anotaciones["Inicio (s)"] == buscar_anotacion[0]) & (self.anotaciones["Duración (s)"] == buscar_anotacion[1]) & (self.anotaciones["Descripcion"] == buscar_anotacion[2])
            return self.anotaciones[buscar]

    def save(self, nombre):
        """Guarda las anotaciones en un archivo .csv
            Args:
                nombre : Nombre con el que se guardara el archivo"""
        return self.anotaciones.to_csv(f"{nombre}.csv")
    
    def load(self, archivo):
        """Carga las anotaciones desde un archivo .csv
            Args:
                archivo : Nombre del archivo que se quiere cargar
            Devuelve el dataframe con los datos del archivo csv"""
        
        anotacion = pd.read_csv(archivo)
        return anotacion

In [10]:
inicio = [5.0, 12.5, 20.0]
duracion = [2.0, 3.0, 3.5]
descripcion = ['Inicio_Experimento', 'Evento_1', 'Evento_2']

In [11]:
anotaciones = Anotaciones(onset=inicio, duration=duracion, description=descripcion)

In [12]:
anotaciones.get_annotations()

,Inicio (s),Duración (s),Descripcion
0,5.0,2.0,Inicio_Experimento
1,12.5,3.0,Evento_1
2,20.0,3.5,Evento_2


In [13]:
nueva_anotacion = [3, 4, "Evento 3"]
anotaciones.add(nueva_anotacion)

True

In [14]:
anotaciones.get_annotations()

,Inicio (s),Duración (s),Descripcion
0,5.0,2.0,Inicio_Experimento
1,12.5,3.0,Evento_1
2,20.0,3.5,Evento_2
3,3.0,4.0,Evento 3


In [15]:
anotaciones.remove(nueva_anotacion)

True

In [16]:
anotaciones.get_annotations()

,Inicio (s),Duración (s),Descripcion
0,5.0,2.0,Inicio_Experimento
1,12.5,3.0,Evento_1
2,20.0,3.5,Evento_2


In [17]:
anotaciones.find(nueva_anotacion)

,Inicio (s),Duración (s),Descripcion


In [18]:
anotaciones.save("Anotaciones")

In [19]:
anotaciones.load("Anotaciones.csv")

,Unnamed: 0,Inicio (s),Duración (s),Descripcion
0,0,5.0,2.0,Inicio_Experimento
1,1,12.5,3.0,Evento_1
2,2,20.0,3.5,Evento_2
